In [1]:
import torch
import json
import sys
from pathlib import Path

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

with open(DATA_DIR / "vocab.json", encoding="utf-8") as f:
    word_to_id = json.load(f)

id_to_word = {idx: word for word, idx in word_to_id.items()}

VOCAB_SIZE = len(word_to_id)
PAD_ID = word_to_id["<pad>"]
UNK_ID = word_to_id["<unk>"]

CONTEXT_LENGTH = 32
DEVICE = torch.device("cpu")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from language_models import LSTMLanguageModel


def load_lstm_checkpoint(filename):
    model = LSTMLanguageModel(
        vocab_size=VOCAB_SIZE,
        embedding_dim=100,
        hidden_dim=128,
        num_layers=1,
        pad_id=PAD_ID
    ).to(DEVICE)

    checkpoint = torch.load(
        MODELS_DIR / filename,
        map_location=DEVICE,
        weights_only=True
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return model


models = {
    "Random Trainable": load_lstm_checkpoint("lstm_best.pt"),
    "Frozen Skip-gram": load_lstm_checkpoint("lstm_skipgram_best.pt"),
    "Frozen GloVe": load_lstm_checkpoint("lstm_glove_best.pt")
}

print("All three LSTM variants loaded successfully!")

All three LSTM variants loaded successfully!


In [3]:
@torch.no_grad()
def generate_lstm(
    model,
    prompt,
    max_new_tokens=30,
    temperature=1.0,
    top_k=20,
    seed=42
):
    model.eval()

    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(seed)

    tokens = prompt.lower().split()

    generated_ids = [
        word_to_id.get(token, UNK_ID)
        for token in tokens
    ]

    for _ in range(max_new_tokens):

        # Use the last 32 tokens
        input_ids = generated_ids[-CONTEXT_LENGTH:]

        x = torch.tensor(
            [input_ids],
            dtype=torch.long,
            device=DEVICE
        )

        # Forward pass
        logits = model(x)

        # Get next-token logits
        next_logits = logits[0, -1, :].clone()

        # Exclude padding
        next_logits[PAD_ID] = float("-inf")

        # Temperature scaling
        next_logits = next_logits / temperature

        # Top-k filtering
        if top_k is not None:
            values, indices = torch.topk(
                next_logits,
                k=top_k
            )

            filtered_logits = torch.full_like(
                next_logits,
                float("-inf")
            )

            filtered_logits[indices] = values
            next_logits = filtered_logits

        # Convert logits into probabilities
        probs = torch.softmax(next_logits, dim=-1)

        # Sample next token
        next_id = torch.multinomial(
            probs,
            num_samples=1,
            generator=generator
        ).item()

        generated_ids.append(next_id)

    return " ".join(
        id_to_word[idx] for idx in generated_ids
    )

In [4]:
prompts = [
    "the game",
    "in the",
    "the history of"
]

generated_samples = {}

for model_name, model in models.items():

    generated_samples[model_name] = {}

    print("\n" + "=" * 70)
    print("MODEL:", model_name)
    print("=" * 70)

    for prompt in prompts:

        text = generate_lstm(
            model=model,
            prompt=prompt,
            max_new_tokens=30,
            temperature=1.0,
            top_k=20,
            seed=42
        )

        generated_samples[model_name][prompt] = text

        print(f"\nPrompt: {prompt}")
        print(text)


MODEL: Random Trainable

Prompt: the game
the game , " it was never thought that her " was a <unk> . " = = = = = <unk> and career = = = " the <unk> , '

Prompt: in the
in the <unk> . it was also to get at the end of <unk> . the <unk> was a <unk> in <unk> , which he had to <unk> the series , but

Prompt: the history of
the history of the first season of <unk> to the end of the 20th century . the <unk> was a <unk> in <unk> , which he had to <unk> the series , but

MODEL: Frozen Skip-gram

Prompt: the game
the game , where he was first to get at the end of <unk> . the <unk> was a first in part of the same time , it had a result that

Prompt: in the
in the <unk> . it was also to be used by the <unk> and <unk> . = = = = = <unk> = = = = = <unk> and other = =

Prompt: the history of
the history of the first world of <unk> , and the first world war war . the general was established by <unk> <unk> , a more than 60 % of a year that

MODEL: Frozen GloVe

Prompt: the game
the game , " it was rele

In [5]:
with open(
    RESULTS_DIR / "embedding_generation_samples.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        generated_samples,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Embedding generation samples saved!")

Embedding generation samples saved!
